# 🔧 Budowanie MLC LLM dla Samsung Exynos (Mali GPU)

Ten notebook zbuduje bibliotekę `libtvm4j_runtime_packed.so` z wsparciem **Vulkan** dla urządzeń Samsung z procesorem Exynos i GPU Mali.

**Czas budowania:** ~15-20 minut (darmowy Colab)

## Jak używać:
1. Kliknij **Runtime → Run all** (Ctrl+F9)
2. Poczekaj aż wszystko się zbuduje
3. Na końcu pobierz plik `libtvm4j_runtime_packed.so`
4. Skopiuj go do projektu MAUI

In [ ]:
#@title 1️⃣ Instalacja zależności
!apt-get update -qq
!apt-get install -y -qq ninja-build cmake
!pip install -q numpy decorator attrs tornado psutil scipy
print("✅ Zależności zainstalowane")

In [ ]:
#@title 2️⃣ Pobranie Android NDK
import os

# NDK r26c - stabilna wersja
NDK_VERSION = "r26c"
NDK_URL = f"https://dl.google.com/android/repository/android-ndk-{NDK_VERSION}-linux.zip"

!wget -q {NDK_URL} -O ndk.zip
!unzip -q ndk.zip
!rm ndk.zip

NDK_PATH = f"/content/android-ndk-{NDK_VERSION}"
os.environ["ANDROID_NDK"] = NDK_PATH
os.environ["TVM_NDK_CC"] = f"{NDK_PATH}/toolchains/llvm/prebuilt/linux-x86_64/bin/aarch64-linux-android24-clang"

print(f"✅ Android NDK {NDK_VERSION} zainstalowany")
print(f"   Ścieżka: {NDK_PATH}")

In [ ]:
#@title 3️⃣ Klonowanie MLC LLM i TVM
%cd /content
!git clone --recursive https://github.com/mlc-ai/mlc-llm.git
%cd mlc-llm
!git submodule update --init --recursive
print("✅ MLC LLM sklonowane")

In [ ]:
#@title 4️⃣ Konfiguracja TVM dla Android + Vulkan
import os

TVM_PATH = "/content/mlc-llm/3rdparty/tvm"
BUILD_PATH = "/content/mlc-llm/build_android"
NDK_PATH = os.environ["ANDROID_NDK"]

os.makedirs(BUILD_PATH, exist_ok=True)

# Konfiguracja CMake dla Android ARM64 z Vulkan
config_cmake = f"""
set(USE_LLVM OFF)
set(USE_CUDA OFF)
set(USE_OPENCL ON)
set(USE_VULKAN ON)
set(USE_OPENGL OFF)
set(USE_METAL OFF)
set(USE_ROCM OFF)

set(USE_GRAPH_EXECUTOR ON)
set(USE_PROFILER ON)
set(USE_LIBBACKTRACE OFF)

set(CMAKE_SYSTEM_NAME Android)
set(CMAKE_SYSTEM_VERSION 24)
set(CMAKE_ANDROID_ARCH_ABI arm64-v8a)
set(CMAKE_ANDROID_NDK {NDK_PATH})
set(CMAKE_ANDROID_STL_TYPE c++_static)
"""

with open(f"{BUILD_PATH}/config.cmake", "w") as f:
    f.write(config_cmake)

print("✅ Konfiguracja TVM zapisana")
print("   Backend: OpenCL + Vulkan")
print("   Target: Android ARM64 (arm64-v8a)")

In [ ]:
#@title 5️⃣ Budowanie TVM Runtime (to zajmie ~10-15 min)
import os

BUILD_PATH = "/content/mlc-llm/build_android"
TVM_PATH = "/content/mlc-llm/3rdparty/tvm"
NDK_PATH = os.environ["ANDROID_NDK"]
TOOLCHAIN = f"{NDK_PATH}/build/cmake/android.toolchain.cmake"

%cd {BUILD_PATH}

# CMake configure
!cmake {TVM_PATH} \
    -DCMAKE_TOOLCHAIN_FILE={TOOLCHAIN} \
    -DCMAKE_BUILD_TYPE=Release \
    -DANDROID_ABI=arm64-v8a \
    -DANDROID_PLATFORM=android-24 \
    -DANDROID_STL=c++_static \
    -DUSE_OPENCL=ON \
    -DUSE_VULKAN=ON \
    -G Ninja

print("\n🔨 Budowanie... (to zajmie ~10-15 minut)")
!ninja -j4

print("\n✅ TVM Runtime zbudowany!")

In [ ]:
#@title 6️⃣ Instalacja MLC LLM Python
%cd /content/mlc-llm
!pip install -q .

# Dodaj TVM do PYTHONPATH
import sys
sys.path.insert(0, "/content/mlc-llm/3rdparty/tvm/python")

print("✅ MLC LLM Python zainstalowany")

In [ ]:
#@title 7️⃣ Kompilacja modeli dla Android/Vulkan
import os

%cd /content/mlc-llm

# Lista modeli do kompilacji
MODELS = [
    ("HF://mlc-ai/Qwen2.5-1.5B-Instruct-q4f16_1-MLC", "qwen2"),
    ("HF://mlc-ai/Phi-3.5-mini-instruct-q4f16_0-MLC", "phi3"),
    ("HF://mlc-ai/gemma-2-2b-it-q4f16_1-MLC", "gemma2"),
]

os.makedirs("/content/compiled_libs", exist_ok=True)

for model_path, model_name in MODELS:
    print(f"\n🔄 Kompilowanie {model_name}...")
    !mlc_llm compile {model_path} \
        --device android \
        --opt O3 \
        -o /content/compiled_libs/{model_name}_android.tar

print("\n✅ Modele skompilowane!")
!ls -la /content/compiled_libs/

In [ ]:
#@title 8️⃣ Budowanie końcowej biblioteki z JNI
import os
import shutil

# Przygotuj katalog android
%cd /content/mlc-llm/android

# Skopiuj skompilowane biblioteki modeli
!mkdir -p /content/mlc-llm/android/build/model_lib
!cp /content/compiled_libs/*.tar /content/mlc-llm/android/build/model_lib/

# Użyj skryptu prepare do zbudowania końcowej biblioteki
NDK_PATH = os.environ["ANDROID_NDK"]

!mkdir -p build && cd build && \
    cmake .. \
    -DCMAKE_TOOLCHAIN_FILE={NDK_PATH}/build/cmake/android.toolchain.cmake \
    -DCMAKE_BUILD_TYPE=Release \
    -DANDROID_ABI=arm64-v8a \
    -DANDROID_PLATFORM=android-24 \
    -DANDROID_STL=c++_static \
    -DUSE_OPENCL=ON \
    -DUSE_VULKAN=ON \
    -DMLC_LLM_INSTALL_STATIC_LIB=ON \
    -G Ninja && \
    ninja

print("\n✅ Biblioteka JNI zbudowana!")

In [ ]:
#@title 9️⃣ Pakowanie i przygotowanie do pobrania
import shutil
from google.colab import files

OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Znajdź i skopiuj bibliotekę
lib_paths = [
    "/content/mlc-llm/android/build/libtvm4j_runtime_packed.so",
    "/content/mlc-llm/build_android/libtvm4j_runtime_packed.so",
    "/content/mlc-llm/build_android/libtvm_runtime.so",
]

for lib_path in lib_paths:
    if os.path.exists(lib_path):
        shutil.copy(lib_path, OUTPUT_DIR)
        print(f"✅ Skopiowano: {os.path.basename(lib_path)}")

# Skopiuj skompilowane modele
!cp /content/compiled_libs/*.tar {OUTPUT_DIR}/

# Utwórz archiwum
!cd /content && tar -czvf mlc_mali_vulkan.tar.gz output/

print("\n" + "="*50)
print("📦 GOTOWE DO POBRANIA!")
print("="*50)
print("\nZawartość pakietu:")
!ls -la {OUTPUT_DIR}/

print("\n⬇️ Kliknij poniżej aby pobrać:")
files.download("/content/mlc_mali_vulkan.tar.gz")

# 📋 Co dalej?

Po pobraniu `mlc_mali_vulkan.tar.gz`:

1. **Rozpakuj archiwum**
2. **Skopiuj `libtvm4j_runtime_packed.so`** do:
   ```
   LLMClient/Platforms/Android/libs/arm64-v8a/
   ```
3. **Przebuduj projekt** w Visual Studio
4. **Zainstaluj na telefonie** i przetestuj!

---
Jeśli coś nie działa, sprawdź logi w Android Studio (Logcat) szukając `MlcLlmBridge` lub `JSONFFIEngine`.